In [3]:
import sys, warnings, os
warnings.filterwarnings('ignore')
sys.path.append('../..')

import importlib
import src.rag.sec_collector as sc
importlib.reload(sc)

from src.utils.config import RAW_DIR
import pandas as pd

# Skip live download — go straight to synthetic data
save_dir = RAW_DIR / 'sec_filings'
os.makedirs(save_dir, exist_ok=True)

# Delete old broken file if exists
old_file = save_dir / 'sec_filings.csv'
if old_file.exists():
    old_file.unlink()
    print('Deleted old broken file')

# Create synthetic filings directly
df = sc.create_synthetic_filings(
    tickers=['AAPL','MSFT','GOOGL','AMZN','META',
             'JPM','GS','BAC','NVDA','TSLA'],
    save_dir=save_dir
)

print(f'\nDone: {len(df)} filings saved')
print(df[['ticker','form_type','date','char_count']].to_string(index=False))

Deleted old broken file
Synthetic filings created: 30 entries
ticker form_type       date  char_count
  AAPL      10-K 2024-01-01        2814
  AAPL      10-Q 2024-04-01        2814
  AAPL      10-Q 2024-07-01        2814
  MSFT      10-K 2024-01-01        2094
  MSFT      10-Q 2024-04-01        2094
  MSFT      10-Q 2024-07-01        2094
 GOOGL      10-K 2024-01-01        1827
 GOOGL      10-Q 2024-04-01        1827
 GOOGL      10-Q 2024-07-01        1827
  AMZN      10-K 2024-01-01        2058
  AMZN      10-Q 2024-04-01        2058
  AMZN      10-Q 2024-07-01        2058
  META      10-K 2024-01-01        2160
  META      10-Q 2024-04-01        2160
  META      10-Q 2024-07-01        2160
   JPM      10-K 2024-01-01        2118
   JPM      10-Q 2024-04-01        2118
   JPM      10-Q 2024-07-01        2118
    GS      10-K 2024-01-01        1782
    GS      10-Q 2024-04-01        1782
    GS      10-Q 2024-07-01        1782
   BAC      10-K 2024-01-01        1887
   BAC      10-Q 2

In [ ]:
import importlib
import src.rag.chunker as chunker_mod
importlib.reload(chunker_mod)
from src.rag.chunker import chunk_filings
chunks = chunk_filings()
print(f'\nTotal chunks: {len(chunks)}')
print(f'\nSample chunk 0:')
print(f'  Ticker:    {chunks[0].metadata["ticker"]}')
print(f'  Form:      {chunks[0].metadata["form_type"]}')
print(f'  Date:      {chunks[0].metadata["date"]}')
print(f'  Length:    {len(chunks[0].page_content)} chars')
print(f'  Preview:   {chunks[0].page_content[:300]}...')

Total chunks created: 150
Average chunk length: 416 chars

Chunks per ticker:
  AAPL: 21 chunks
  AMZN: 15 chunks
  BAC: 12 chunks
  GOOGL: 12 chunks
  GS: 12 chunks
  JPM: 15 chunks
  META: 15 chunks
  MSFT: 15 chunks
  NVDA: 18 chunks
  TSLA: 15 chunks

Total chunks: 150

Sample chunk 0:
  Ticker:    AAPL
  Form:      10-K
  Date:      2024-01-01
  Length:    486 chars
  Preview:   Apple Inc. Annual Report. Apple designs, manufactures
        and markets smartphones, personal computers, tablets, wearables
        and accessories, and sells a variety of related services.
        iPhone revenue represents the largest segment. Risk factors include
        dependence on iPhone sal...


: 

In [ ]:
import importlib
import src.rag.embedder as embedder_mod
importlib.reload(embedder_mod)
from src.rag.embedder import build_faiss_index, load_faiss_index
from src.utils.config import PROCESSED_DIR
index_path = PROCESSED_DIR / 'rag' / 'faiss_index'
if index_path.exists():
    print('FAISS index already exists — loading...')
    vectorstore = load_faiss_index()
    print(f'Index loaded: {vectorstore.index.ntotal} vectors')
else:
    print('Building FAISS index...')
    vectorstore = build_faiss_index(chunks, save=True)
    print(f'Index built: {vectorstore.index.ntotal} vectors')

Building FAISS index...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Embedding model loaded.
Building FAISS index for 150 chunks...
This takes 2-5 minutes on CPU for 1000+ chunks...
  Indexed 100/150 chunks (66%)
  Indexed 150/150 chunks (100%)

FAISS index saved to c:\Users\default.LAPTOP-60DKOJUD\projects\atlas-trading-platform\notebooks\08_rag\..\..\data\processed\rag\faiss_index
Index built: 150 vectors


: 

In [ ]:
import importlib
import src.rag.retriever as ret_mod
importlib.reload(ret_mod)
from src.rag.retriever import retrieve, format_results
# Test 5 different queries across different tickers
test_queries = [
    ('AAPL',  'iPhone demand risks China competition'),
    ('NVDA',  'data center AI chip revenue growth'),
    ('META',  'advertising revenue user engagement'),
    ('TSLA',  'vehicle delivery production capacity'),
    ('JPM',   'credit losses loan defaults recession'),
]
for ticker, query in test_queries:
    print(f'\n{"="*60}')
    print(f'TICKER: {ticker}')
    print(f'QUERY:  {query}')
    print(f'{"="*60}')
    results = retrieve(query, ticker=ticker, k=2,
                        vectorstore=vectorstore)
    if results:
        doc, score = results[0]
        print(f'Top result similarity: {1-score:.3f}')
        print(f'Source: {doc.metadata["form_type"]} ({doc.metadata["date"]})')
        print(f'Preview: {doc.page_content[:300]}...')
    else:
        print('No results found')


TICKER: AAPL
QUERY:  iPhone demand risks China competition
Top result similarity: 0.322
Source: 10-K (2024-01-01)
Preview: Macroeconomic conditions including consumer spending affect demand.Apple Inc. Annual Report. Apple designs, manufactures
        and markets smartphones, personal computers, tablets, wearables
        and accessories, and sells a variety of related services.
        iPhone revenue represents the lar...

TICKER: NVDA
QUERY:  data center AI chip revenue growth
Top result similarity: 0.439
Source: 10-K (2024-01-01)
Preview: infrastructure company. Data Center segment revenue grew significantly
        driven by AI and machine learning workloads. Gaming GPU revenue
        faces cyclical demand. Products include H100 and A100 data center
        GPUs, GeForce gaming GPUs, automotive chips. Risk factors include
        e...

TICKER: META
QUERY:  advertising revenue user engagement
Top result similarity: -0.001
Source: 10-K (2024-01-01)
Preview: engagement growth. Cost 

: 

In [ ]:
import importlib
import src.rag.generator as gen_mod
importlib.reload(gen_mod)
from src.rag.generator import generate_market_brief
# Simulate model signals from Phase 6
model_signals = {
    'AAPL': 'BEARISH — XGBoost 3-modal AUC 0.71, predicts down day',
    'NVDA': 'BULLISH — XGBoost 3-modal AUC 0.68, predicts up day',
    'META': 'NEUTRAL — XGBoost 3-modal AUC 0.57, uncertain',
}
queries = {
    'AAPL': 'iPhone demand risks competition China',
    'NVDA': 'data center AI chip revenue growth outlook',
    'META': 'advertising revenue headwinds regulatory risks',
}
for ticker in ['AAPL', 'NVDA', 'META']:
    print('\n' + '='*60)
    brief = generate_market_brief(
        ticker=ticker,
        query=queries[ticker],
        model_signal=model_signals.get(ticker),
        vectorstore=vectorstore
    )
    print(brief)


ATLAS MARKET BRIEF — AAPL
Query: iPhone demand risks competition China
Model Signal: BEARISH — XGBoost 3-modal AUC 0.71, predicts down day
--------------------------------------------------
Key Findings from SEC Filings:

1. Risk factors include
        dependence on iPhone sales, competition from Samsung and Chinese
        manufacturers, supply chain concentration in Asia, regulatory Macroeconomic conditions including consumer spending affect demand.Apple Inc.

2. Apple designs, manufactures
        and markets smartphones, personal computers, tablets, wearables
        and accessories, and sells a variety of related services.
        iPhone revenue represents the largest segment.

3. Macroeconomic conditions including consumer spending affect demand.Apple Inc.

--------------------------------------------------
Sources:
  - AAPL 10-K (2024-01-01) [similarity: 0.33]
  - AAPL 10-K (2024-01-01) [similarity: 0.33]
  - AAPL 10-Q (2024-04-01) [similarity: 0.33]

ATLAS MARKET BRIEF — NVDA

: 

In [ ]:
import pandas as pd
from src.rag.retriever import retrieve

# Evaluate retrieval quality with known questions
eval_queries = [
    {
        'ticker':   'AAPL',
        'query':    'revenue from iPhone segment',
        'expected': 'iPhone',  # Should appear in top result
    },
    {
        'ticker':   'NVDA',
        'query':    'gaming GPU revenue decline',
        'expected': 'gaming',
    },
    {
        'ticker':   'JPM',
        'query':    'net interest income margin',
        'expected': 'interest',
    },
]

print('RAG RETRIEVAL QUALITY EVALUATION')
print('='*50)

hits = 0
for item in eval_queries:
    results = retrieve(
        item['query'],
        ticker=item['ticker'],
        k=3,
        vectorstore=vectorstore
    )
    
    if results:
        top_text = results[0][0].page_content.lower()
        hit = item['expected'].lower() in top_text
        hits += int(hit)
        status = 'HIT' if hit else 'MISS'
        score  = 1 - results[0][1]
        print(f'{status} [{item["ticker"]}] {item["query"][:40]}'
              f' | similarity={score:.3f}')
    else:
        print(f'MISS [{item["ticker"]}] {item["query"][:40]}'
              f' | no results')

print(f'\nPrecision: {hits}/{len(eval_queries)} = '
      f'{hits/len(eval_queries)*100:.0f}%')
print('\nNote: Higher similarity score = more relevant result')
print('Good RAG: similarity > 0.7 for domain-specific queries')

RAG RETRIEVAL QUALITY EVALUATION
HIT [AAPL] revenue from iPhone segment | similarity=0.249
HIT [NVDA] gaming GPU revenue decline | similarity=0.227
HIT [JPM] net interest income margin | similarity=0.088

Precision: 3/3 = 100%

Note: Higher similarity score = more relevant result
Good RAG: similarity > 0.7 for domain-specific queries


: 

In [3]:
import sys, warnings, os
warnings.filterwarnings('ignore')
sys.path.append('../..')

# Force reload ALL rag modules — clears kernel cache
import importlib
import src.rag.embedder   as emb_mod
import src.rag.retriever  as ret_mod
import src.rag.generator  as gen_mod
importlib.reload(emb_mod)
importlib.reload(ret_mod)
importlib.reload(gen_mod)
from src.rag.embedder  import load_faiss_index
from src.rag.generator import generate_market_brief

import joblib
import pandas as pd
import numpy as np
from src.macro.macro_features      import add_macro_to_splits
from src.sentiment.sentiment_features import add_sentiment_to_splits
from src.models.data_loader        import load_splits

# ── Load features ───────────────────────────────────────────────
print('Loading features...')
X_train_73, y_train, X_val, y_val, X_test_73, y_test = (
    add_macro_to_splits('AAPL')
)
X_train_54, _, _, _, X_test_54, _ = add_sentiment_to_splits('AAPL')
X_train_43, _, _, _, X_test_43, _ = load_splits('AAPL')
y_train.index = pd.to_datetime(y_train.index)

# ── Find best available model ───────────────────────────────────
model_dir = '../../experiments/models/'
print(f'\nAvailable models:')
for f in sorted(os.listdir(model_dir)):
    if 'AAPL' in f and f.endswith('.pkl'):
        print(f'  {f}')

model_feature_map = {
    'xgboost_AAPL_3modal.pkl':    (X_test_73, 'Phase 6 — 3modal'),
    'xgboost_AAPL_sentiment.pkl': (X_test_54, 'Phase 5 — sentiment'),
    'xgboost_AAPL.pkl':           (X_test_43, 'Phase 4 — base'),
}

xgb_model   = None
X_test_use  = None
model_label = None

for model_name, (X_test_candidate, label) in model_feature_map.items():
    model_path = model_dir + model_name
    if os.path.exists(model_path):
        try:
            m    = joblib.load(model_path)
            _    = m.predict_proba(X_test_candidate.iloc[[-1]])
            xgb_model   = m
            X_test_use  = X_test_candidate
            model_label = label
            print(f'\nUsing: {model_name} ({label})')
            break
        except Exception as e:
            print(f'  Skip {model_name}: {str(e)[:50]}')
            continue

if xgb_model is None:
    raise ValueError('No compatible model found')

# ── Make prediction ─────────────────────────────────────────────
latest_features = X_test_use.iloc[[-1]]
latest_date     = X_test_use.index[-1]
pred_proba      = xgb_model.predict_proba(latest_features)[0]
pred_class      = 'BULLISH' if pred_proba[1] > 0.5 else 'BEARISH'
confidence      = max(pred_proba)

print(f'\n{"="*50}')
print(f'ATLAS PREDICTION')
print(f'{"="*50}')
print(f'Model:      {model_label}')
print(f'Ticker:     AAPL')
print(f'Date:       {str(latest_date)[:10]}')
print(f'Signal:     {pred_class}')
print(f'Confidence: {confidence:.1%}')
print(f'Up prob:    {pred_proba[1]:.3f}')
print(f'Down prob:  {pred_proba[0]:.3f}')

# ── RAG explanation ─────────────────────────────────────────────
query = ('revenue risks competition market headwinds decline'
         if pred_class == 'BEARISH'
         else 'revenue growth strong demand market opportunity')

print(f'\n{"="*50}')
print(f'ATLAS RAG EXPLANATION')
print(f'{"="*50}')
vs    = load_faiss_index()
brief = generate_market_brief(
    ticker='AAPL',
    query=query,
    model_signal=f'{pred_class} — {confidence:.1%} | {model_label}',
    vectorstore=vs
)
print(brief)

Loading features...
AAPL — Train:1762 Val:124 Test:622
AAPL: 11 sentiment proxy features built
AAPL: 43 → 54 features
AAPL: 19 macro features aligned
  Macro date range: 2016-01-04 00:00:00 to 2025-12-22 00:00:00

AAPL feature expansion:
  Technical+Sentiment: 54 features
  After adding macro:  73 features
  Macro features added: 19
AAPL — Train:1762 Val:124 Test:622
AAPL: 11 sentiment proxy features built
AAPL: 43 → 54 features
AAPL — Train:1762 Val:124 Test:622

Available models:
  lgbm_AAPL.pkl
  xgboost_AAPL.pkl

Using: xgboost_AAPL.pkl (Phase 4 — base)

ATLAS PREDICTION
Model:      Phase 4 — base
Ticker:     AAPL
Date:       2025-12-22
Signal:     BEARISH
Confidence: 84.2%
Up prob:    0.158
Down prob:  0.842

ATLAS RAG EXPLANATION
FAISS index loaded — 150 vectors
ATLAS MARKET BRIEF — AAPL
Query: revenue risks competition market headwinds decline
Model Signal: BEARISH — 84.2% | Phase 4 — base
--------------------------------------------------
Key Findings from SEC Filings:

1. Chin

In [4]:
# Show complete Cell 7 output without truncation
import sys, warnings, os
warnings.filterwarnings('ignore')
sys.path.append('../..')

import importlib
import src.rag.embedder  as emb_mod
import src.rag.generator as gen_mod
importlib.reload(emb_mod)
importlib.reload(gen_mod)
from src.rag.embedder  import load_faiss_index
from src.rag.generator import generate_market_brief

import joblib, pandas as pd
from src.models.data_loader import load_splits

# Load Phase 4 model with correct 43 features
X_train, y_train, X_val, y_val, X_test, y_test = load_splits('AAPL')
y_train.index = pd.to_datetime(y_train.index)

xgb_model = joblib.load('../../experiments/models/xgboost_AAPL.pkl')

latest_features = X_test.iloc[[-1]]
latest_date     = X_test.index[-1]
pred_proba      = xgb_model.predict_proba(latest_features)[0]
pred_class      = 'BULLISH' if pred_proba[1] > 0.5 else 'BEARISH'
confidence      = max(pred_proba)

print('='*50)
print('ATLAS PREDICTION')
print('='*50)
print(f'Ticker:     AAPL')
print(f'Date:       {str(latest_date)[:10]}')
print(f'Signal:     {pred_class}')
print(f'Confidence: {confidence:.1%}')
print(f'Up prob:    {pred_proba[1]:.3f}')
print(f'Down prob:  {pred_proba[0]:.3f}')

query = ('revenue risks competition market headwinds decline'
         if pred_class == 'BEARISH'
         else 'revenue growth strong demand market opportunity')

vs    = load_faiss_index()
brief = generate_market_brief(
    ticker='AAPL',
    query=query,
    model_signal=f'{pred_class} — {confidence:.1%} | Phase 4 base',
    vectorstore=vs
)

# Print without truncation
print()
for line in brief.split('\n'):
    print(line)

AAPL — Train:1762 Val:124 Test:622
ATLAS PREDICTION
Ticker:     AAPL
Date:       2025-12-22
Signal:     BEARISH
Confidence: 84.2%
Up prob:    0.158
Down prob:  0.842
FAISS index loaded — 150 vectors

ATLAS MARKET BRIEF — AAPL
Query: revenue risks competition market headwinds decline
Model Signal: BEARISH — 84.2% | Phase 4 base
--------------------------------------------------
Key Findings from SEC Filings:

1. China market faces headwinds
        from local competition and geopolitical risks.

2. Management discussion: revenue growth
        driven by services segment including App Store, Apple Music, iCloud.
        Operating income margins remain strong.

3. driven by services segment including App Store, Apple Music, iCloud.
        Operating income margins remain strong.

4. R&D investment in AI and
        augmented reality.

5. Data privacy regulations present compliance costs.

--------------------------------------------------
Sources:
  - AAPL 10-K (2024-01-01) [similarity: -